## Required Configuration
You **must** provide your AWS Secrets Manager secret name, AWS region, and connection mode below before running this notebook. The notebook will not continue if `secret_name` or `aws_region` are left as empty strings.

In [ ]:
# REQUIRED: Update these values before running the notebook
secret_name = ""  # e.g. "docdb/my-cluster-secret"
aws_region = ""   # e.g. "us-east-1"

# Set to 'y' if connecting locally through a bastion host, 'n' otherwise
is_bastion = 'n'

assert secret_name, "secret_name must be provided"
assert aws_region, "aws_region must be provided"
print(f"Configuration: secret_name={secret_name!r}, aws_region={aws_region!r}, is_bastion={is_bastion!r}")

## 1. Install Dependencies
Install the required Python packages for connectivity, embeddings, LLM integration, and UI.

In [ ]:
import sys
import os
os.environ.update({
    'USER_AGENT': 'DocumentDB_Chatbot/3.0'
})
!{sys.executable} -m pip install -q -q pymongo gradio beautifulsoup4 pypdf langchain-aws langchain langchain-community html2text tenacity 2>/dev/null

print("Requirements installed")

## 2. Import Libraries
Core imports for document processing, vector search, LLM/embeddings, and chat interface.

In [ ]:
# Import libraries
import json
import urllib.request 
import time
import threading
import logging
import numpy as np
from IPython.display import display, Markdown
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache, wraps
from typing import List

import warnings
import boto3
import gradio as gr
from botocore.exceptions import ClientError
from bs4 import BeautifulSoup
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure, OperationFailure
from tenacity import (
    retry, 
    stop_after_attempt, 
    wait_exponential, 
    retry_if_exception_type
)
from tqdm import tqdm

# Langchain
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_aws import BedrockEmbeddings, ChatBedrock, ChatBedrockConverse
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.document_transformers import Html2TextTransformer
from langchain_community.vectorstores import DocumentDBVectorSearch

print("Imports complete")

## 3. Resilience Utilities
Rate limiter and circuit breaker decorators for Bedrock API limits.

When calling Bedrock at scale, API rate limits are real. This section includes two resilience patterns:

- **RateLimiter** — Enforces a maximum number of calls within a rolling time window. Without it, concurrent users can trigger `ThrottlingException` errors. With it, excess requests wait briefly instead of failing outright.

- **CircuitBreaker** — After a threshold of consecutive failures, stops calling the failing service entirely and fails fast. Without it, every request during an outage hangs for the full socket timeout before returning an error. With it, requests after the threshold are rejected immediately, and a probe request is attempted after a cooldown period to check if the service has recovered.

In [ ]:
# RateLimiter
## Enforces a maximum number of calls within a rolling time window.
class RateLimiter:
    def __init__(self, max_requests, time_window):
        self.max_requests = max_requests
        self.time_window = time_window
        self.requests = []
        self.lock = threading.Lock()

    def __call__(self, func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            with self.lock:
                now = time.time()
                self.requests = [req for req in self.requests if req > now - self.time_window]

                if len(self.requests) >= self.max_requests:
                    # Sleep until the oldest request is far enough in the past that dropping it brings us back under the limit
                    sleep_time = self.requests[0] - (now - self.time_window)
                    time.sleep(max(0, sleep_time))

                self.requests.append(now)
            return func(*args, **kwargs)
        return wrapper

# CircuitBreaker
# ## Stops calling a failing dependency after a threshold of consecutive errors, then automatically retries after a timeout
class CircuitBreaker:
    def __init__(self, failure_threshold=5, reset_timeout=60):
        self.failure_count = 0
        self.failure_threshold = failure_threshold
        self.reset_timeout = reset_timeout
        self.last_failure_time = 0
        self.is_open = False
        self.lock = threading.Lock()

    def __call__(self, func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            with self.lock:
                if self.is_open:
                    if time.time() - self.last_failure_time > self.reset_timeout:
                        # Timeout has elapsed - allow one probe request through
                        self.is_open = False
                        self.failure_count = 0
                    else:
                        # Circuit is open - fail immediately without calling Bedrock
                        raise Exception("Circuit breaker is open")

            try:
                result = func(*args, **kwargs)
                with self.lock:
                    # Successful call - reset the failure counter
                    self.failure_count = 0
                return result
            except Exception as e:
                with self.lock:
                    self.failure_count += 1
                    self.last_failure_time = time.time()
                    if self.failure_count >= self.failure_threshold:
                        # Threshold crossed - open the circuit
                        self.is_open = True
                raise e
        return wrapper

print("Class load complete")

## 4. Connect to DocumentDB
Retrieve credentials from Secrets Manager, connect to the cluster, and create an HNSW vector index on the collection.

DocumentDB supports two vector index types:

- **IVFFlat** divides data into clusters and searches only the relevant ones. Fast searches, but requires pre-training with your full dataset. Adding significantly different data later can degrade quality because new entries get placed into clusters that weren't designed for them. May require periodic index rebuilds.

- **HNSW** builds a multi-level graph where each data point connects to nearby neighbors at different scales. No pre-training needed — new data dynamically forms connections based on its actual characteristics. Higher memory usage, but typically faster and more accurate searches, especially for high-dimensional data. Ideal for growing datasets.

In [ ]:
db_name = 'rag'
coll_name = 'chatbot'

# Vector dimensions must match embedding model output (Titan v2 = 1024)
vector_dimensions = 1024

# HNSW index parameters
hnsw_m = 16
hnsw_efconstruction = 64
tls_file = os.path.abspath("global-bundle.pem")

def get_secret():
    """Retrieve and parse secret from AWS Secrets Manager"""
    try:
        session = boto3.session.Session()
        secrets_client = session.client(
            service_name='secretsmanager',
            region_name=aws_region
        )

        response = secrets_client.get_secret_value(SecretId=secret_name)
        return json.loads(response['SecretString'])
    except ClientError as e:
        print(f"Failed to retrieve secret: {e}")
        raise

def create_vector_index(collection):
    """Create HNSW vector index if it doesn't exist"""
    try:
        existing_indexes = collection.list_indexes()
        if "hnsw" not in [idx["name"] for idx in existing_indexes]:
            
            collection.create_index(
                [("vectorContent", "vector")],
                vectorOptions={
                    "type": "hnsw",
                    "similarity": "cosine",
                    "dimensions": vector_dimensions,
                    "m": hnsw_m,
                    "efConstruction": hnsw_efconstruction
                },
                name="hnsw"
            )
            print("Vector index created successfully")
        else:
            print("Vector index already exists")
    except OperationFailure as e:
        print(f"Failed to create index: {e}")
        raise

def check_data_exists(collection):
    """Check if there is an existing collection with embedding data"""
    try:
        count = collection.count_documents({})
        has_embeddings = collection.count_documents({"vectorContent": {"$exists": True}}) > 0
        return count > 0 and has_embeddings
    except Exception as e:
        print(f"Error checking data: {e}")
        return False

client = None
try:
    secret = get_secret()

    docdb_user = secret['username']
    docdb_pass = secret['password']
    docdb_cluster = secret['host']
    docdb_port = secret['port']

    connection_kwargs = {
        'host': '127.0.0.1' if is_bastion == 'y' else docdb_cluster,
        'port': int(docdb_port),
        'username': docdb_user,
        'password': docdb_pass,
        'tls': True,
        'tlsCAFile': tls_file,
        'tlsAllowInvalidHostnames': is_bastion == 'y',
        'replicaSet': 'rs0',
        'readPreference': 'secondaryPreferred',
        'retryWrites': False,
        'maxPoolSize': 10,
        'minPoolSize': 2,
        'serverSelectionTimeoutMS': 5000,
        'connectTimeoutMS': 2000,
        'socketTimeoutMS': 30000,
        'directConnection': is_bastion == 'y'
    }

    warnings.filterwarnings('ignore', message='.*DocumentDB cluster.*')
    client = MongoClient(**connection_kwargs)
    
    client.admin.command('ping')
    print("Successfully connected to DocumentDB")

    database = client[db_name]
    collection = database[coll_name]

    create_vector_index(collection)

except ConnectionFailure as e:
    print(f"Failed to connect to DocumentDB: {e}")
    raise
except Exception as e:
    print(f"An error occurred: {e}")
    raise

# Initialize Amazon Bedrock for embeddings
# Titan Embed Text v2 produces 1024-dimensional vectors (matches vector_dimensions parameter above)
try:
    bedrock_client = boto3.client('bedrock-runtime', region_name=aws_region)
    embeddings = BedrockEmbeddings(
        model_id="amazon.titan-embed-text-v2:0",
        client=bedrock_client
    )
    print("Successfully initialized Bedrock client and embeddings")
except Exception as e:
    print(f"Error initializing Bedrock client: {e}")
    raise

## 5. Load and Chunk Documents
Load PDFs, crawl AWS blog posts, and fetch DocumentDB documentation pages. Split all content into chunks for embedding.

In [ ]:
# Chunking parameters - critical for RAG quality
## chunk_size - Characters per chunk (balance context vs token limits)
## chunk_overlap - Overlap prevents losing context at chunk boundaries
chunk_size = 1000
chunk_overlap = 200
text_separators = ["\n\n", "\n"]
index_name = "hnsw"

def load_and_chunk_pdfs(pdf_directory: str) -> List[Document]:
    if not os.path.isdir(pdf_directory):
        raise ValueError(f"Invalid directory path: {pdf_directory}")

    try:
        print(f"Loading PDFs from {pdf_directory}")
        pdf_loader = PyPDFDirectoryLoader(pdf_directory)
        pdf_docs = pdf_loader.load()

        print(f"Processing {len(pdf_docs)} documents")
        
        # RecursiveCharacterTextSplitter splits on natural boundaries
        ## Tries separators in order: paragraphs -> lines -> sentences
        text_splitter = RecursiveCharacterTextSplitter(
            separators=text_separators,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            is_separator_regex=False
        )

        # Each chunk becomes a document with its own embedding
        pdf_chunks = text_splitter.split_documents(pdf_docs)
        print(f"Created {len(pdf_chunks)} chunks")

        return pdf_chunks

    except FileNotFoundError:
        print(f"Directory not found: {pdf_directory}")
        return []
    except Exception as e:
        print(f"Error processing PDFs: {e}")
        return []

def extract_and_chunk_content(url: str) -> List[Document]:
    blogURLs = []
    pageURL = url
    all_chunks = []

    # Crawl pagination and collect blog URLs
    print("Collecting blog URLs...")
    while pageURL:
        try:
            html = urllib.request.urlopen(pageURL, timeout=30)
            htmlParse = BeautifulSoup(html, 'html.parser') 
            blogLinks = htmlParse.find_all('h2', attrs={'class': 'lb-bold blog-post-title'})

            for link in blogLinks:
                anchor = link.find('a')
                if anchor and anchor.get('href'):
                    blogURLs.append(anchor['href'])

            next_link = htmlParse.find('link', attrs={'rel': 'next'})
            pageURL = next_link['href'] if next_link else ""

        except Exception as e:
            print(f"Error crawling page {pageURL}: {e}")
            break

    # Process each blog URL
    print(f"Processing {len(blogURLs)} blog posts...")
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " ", ""],
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False
    )
    for blog_url in blogURLs:
        try:
            response = urllib.request.urlopen(blog_url, timeout=30)
            soup = BeautifulSoup(response, 'html.parser')

            content_section = soup.find('section', {
                'class': 'blog-post-content lb-rtxt',
                'property': 'articleBody'
            })

            if content_section:
                author_section = content_section.find('h3', string='About the Authors')
                if author_section:
                    for elem in author_section.find_all_next():
                        elem.decompose()
                    author_section.decompose()

                text_content = content_section.get_text(separator='\n\n', strip=True)

                chunks = text_splitter.create_documents(
                    texts=[text_content],
                    metadatas=[{"source": blog_url}]
                )

                all_chunks.extend(chunks)

        except Exception as e:
            print(f"Error processing {blog_url}: {e}")

    print(f"Created {len(all_chunks)} chunks from blog posts")
    return all_chunks

def process_docdb_pages(urls: List[str]) -> List[Document]:
    try:
        print(f"Processing {len(urls)} DocumentDB pages...")
        docdb_loader = AsyncHtmlLoader(urls)
        docdb_doc = docdb_loader.load()

        html2text = Html2TextTransformer()
        docs_transformed = html2text.transform_documents(docdb_doc)

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

        docdb_chunks = text_splitter.split_documents(docs_transformed)
        print(f"Created {len(docdb_chunks)} chunks from DocumentDB pages")
        return docdb_chunks

    except Exception as e:
        print(f"Error processing DocumentDB pages: {e}")
        return []

data_exists = check_data_exists(collection)

if data_exists:
    print("Data already exists in collection. Skipping data loading.")
else:
    print("No data found. Loading documents...")
    
    # Process PDF files
    pdf_dir = os.getcwd()
    
    blog_url = "https://aws.amazon.com/blogs/database/category/database/amazon-document-db/"
    docdb_urls = [
        "https://aws.amazon.com/documentdb/pricing/",
        "https://aws.amazon.com/documentdb/faqs/",
        "https://aws.amazon.com/documentdb/features/"
    ]

    with ThreadPoolExecutor(max_workers=3) as executor:
        pdf_future = executor.submit(load_and_chunk_pdfs, pdf_dir)
        web_future = executor.submit(extract_and_chunk_content, blog_url)
        docdb_future = executor.submit(process_docdb_pages, docdb_urls)

        pdf_chunks = pdf_future.result()
        web_chunks = web_future.result()
        docdb_chunks = docdb_future.result()

    print("Document loading and chunking complete.")

## 6. Generate Embeddings and Insert
Embed document chunks using Amazon Titan and batch insert them into DocumentDB with vector content.

In [ ]:
logging.basicConfig(
    level=logging.WARNING,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=4, max=10)
)
def _insert_batch(batch_docs):
    collection.insert_many(batch_docs, ordered=False)

def insert_documents_batch(documents: List[Document], batch_size: int = 100) -> None:
    total_batches = (len(documents) + batch_size - 1) // batch_size
    logger.info(f"Starting batch insertion of {len(documents)} documents")

    try:
        for i in tqdm(range(0, len(documents), batch_size), total=total_batches, desc="Processing batches"):
            batch = documents[i:i + batch_size]
            
            # Generate embeddings and insert
            texts = []
            valid_docs = []
            
            for doc in batch:
                text = doc.page_content.replace('\n', ' ').strip()

                # Remove title and page_label
                title = doc.metadata.get('title', '')
                page_label = doc.metadata.get('page_label', '')

                if title:
                    normalized_title = title.replace(' - ', ' ').replace('-', ' ').strip()
                    if text.startswith(normalized_title):
                        text = text[len(normalized_title):]

                if page_label and text.endswith(' ' + page_label):
                    text = text[:-len(' ' + page_label)]

                text = text.strip()

                # Skip table of contents pages and empty documents
                is_roman_numeral = (page_label and
                                len(page_label) <= 10 and
                                all(c in 'ivxlc' for c in page_label.lower()) and
                                any(c in page_label.lower() for c in 'ivxlc'))
                
                if len(text) > 10 and not is_roman_numeral:
                    texts.append(text)
                    valid_docs.append(doc)
            
            if not texts:
                continue
            
            # Generate embeddings - converts text to 1024 dimension vectors
            vectors = embeddings.embed_documents(texts)
            
            # Document structure stored in DocumentDB:
            ## textContent - original text for retrieval
            ## vectorContent - embedding for similarity search
            ## metadata - source info for context
            docs_to_insert = [
                {
                    "textContent": text,
                    "vectorContent": vector,
                    "metadata": {k: v for k, v in doc.metadata.items() 
                               if k not in ['producer', 'creator', 'keywords']}
                }
                for doc, text, vector in zip(valid_docs, texts, vectors)
            ]
            
            _insert_batch(docs_to_insert)

    except Exception as e:
        logger.error(f"Error inserting batch of documents: {str(e)}")
        raise

def process_all_documents(pdf_chunks: List[Document], web_chunks: List[Document], docdb_chunks: List[Document], batch_size: int = 100) -> None:
    try:
        all_chunks = pdf_chunks + web_chunks + docdb_chunks
        logger.info(f"Processing {len(all_chunks)} documents in optimized batch mode")

        if not all_chunks:
            logger.warning("No documents to process")
            return

        insert_documents_batch(all_chunks, batch_size=batch_size)
        logger.info("Optimized processing completed successfully")

    except Exception as e:
        logger.error(f"Error processing documents: {str(e)}")
        raise

if data_exists:
    print("Embedded data already exists in collection. Skipping data embedding.")
else:
    print("Embedding documents...")
    process_all_documents(pdf_chunks, web_chunks, docdb_chunks, batch_size=100)
    print("Document processing and embedding complete.")

## 7. Single-Store vs. Multi-Store RAG Architecture
This section compares performance of storing embeddings and text in separate repositories vs. a single collection

Most RAG architectures require two separate databases — a vector store for similarity search and a document store for source text. Retrieval then requires two round trips and a manual join: query the vector store to get document IDs, then fetch each ID from the document store. This also means every write must be wrapped in a transaction across both stores. If the vector insert succeeds but the text insert fails, the vector exists with no corresponding content — a silent data consistency bug that's undetectable at query time.

With DocumentDB, vectors and source text live in the same document. Retrieval is a single `$search` aggregation that returns everything in one hop. Writes are a single insert — no transaction needed, nothing to fall out of sync.

This demo makes use of documents already processed in previous cells to create 3 new collections:
- `multi_embed` - embeddings only
- `multi_doc` - text only
- `single_arch` - both together

`multi_embed` and `multi_doc` demonstrate storing text and embeddings in different locations, joining later by a common `_id`
`single_arch` shows using a single collection for both

The end result compares the insertion time and average query response time for each approach. 

In [ ]:
sample_size = 1000
n_runs = 5

print(f"Loading {sample_size} documents from collection...")
sample_docs = list(collection.find(
    {"vectorContent": {"$exists": True}},
    {"_id": 1, "textContent": 1, "vectorContent": 1}
).limit(sample_size))
print(f"  Loaded {len(sample_docs)} documents\n")

query_vector = embeddings.embed_query("How does vector search work in DocumentDB?")

for col_name in ["multi_embed", "multi_doc", "single_arch"]:
    database[col_name].drop()

multi_embed = database.create_collection("multi_embed")
multi_doc   = database.create_collection("multi_doc")
single_arch = database.create_collection("single_arch")

embed_docs  = [{"_id": d["_id"], "vectorContent": d["vectorContent"]} for d in sample_docs]
text_docs   = [{"_id": d["_id"], "textContent": d.get("textContent", "")} for d in sample_docs]
single_docs = [{"_id": d["_id"], "textContent": d.get("textContent", ""),
                "vectorContent": d["vectorContent"]} for d in sample_docs]

# Mulsit-store insertion
print("-" * 60)
print(f"Multi-store insertion")

t0 = time.perf_counter()
with client.start_session() as session:
    with session.start_transaction():
        multi_embed.insert_many(embed_docs, session=session)
        multi_doc.insert_many(text_docs,   session=session)
multi_insert_ms = (time.perf_counter() - t0) * 1000

print(f"  Time -> {multi_insert_ms:.1f}ms")

multi_embed.create_index(
    [("vectorContent", "vector")],
    vectorOptions={"type": "hnsw", "similarity": "cosine",
                   "dimensions": vector_dimensions, "m": hnsw_m, "efConstruction": hnsw_efconstruction},
    name="hnsw"
)

# Multi-store retrieval
print("\n" + "-" * 60)
print(f"Multi-store retrieval  (avg of {n_runs} runs)")

multi_times = []
for _ in range(n_runs):
    t0 = time.perf_counter()
    vec_results = list(multi_embed.aggregate([
        {"$search": {"vectorSearch": {"vector": query_vector, "path": "vectorContent",
                                      "similarity": "cosine", "k": 5, "efSearch": hnsw_efconstruction}}},
        {"$project": {"_id": 1}}
    ]))
    ids = [doc["_id"] for doc in vec_results]
    doc_results = list(multi_doc.find({"_id": {"$in": ids}}, {"textContent": 1}))
    multi_times.append((time.perf_counter() - t0) * 1000)

multi_query_ms = sum(multi_times) / n_runs
print(f"  Avg time: {multi_query_ms:.1f}ms")

# Single-store insertion
print("\n" + "-" * 60)
print(f"Single-store insertion")

t0 = time.perf_counter()
single_arch.insert_many(single_docs)
single_insert_ms = (time.perf_counter() - t0) * 1000

print(f"  Time: {single_insert_ms:.1f}ms")

single_arch.create_index(
    [("vectorContent", "vector")],
    vectorOptions={"type": "hnsw", "similarity": "cosine",
                   "dimensions": vector_dimensions, "m": hnsw_m, "efConstruction": 64},
    name="hnsw"
)

# Single-store retrieval
print("\n" + "-" * 60)
print(f"Single-store retrieval  (avg of {n_runs} runs)")

single_times = []
for _ in range(n_runs):
    t0 = time.perf_counter()
    single_results = list(single_arch.aggregate([
        {"$search": {"vectorSearch": {"vector": query_vector, "path": "vectorContent",
                                      "similarity": "cosine", "k": 5, "efSearch": 64}}},
        {"$project": {"textContent": 1}}
    ]))
    single_times.append((time.perf_counter() - t0) * 1000)

single_query_ms = sum(single_times) / n_runs
print(f"  Avg time: {single_query_ms:.1f}ms")

insert_factor = multi_insert_ms / max(single_insert_ms, 0.001)
query_factor  = multi_query_ms  / max(single_query_ms,  0.001)

print("\n" + "=" * 60)
print("Summary")
print("=" * 60)
print(f"  {'':22} {'Multi-store':>12} {'Single-store':>13} {'Diff':>8}")
print(f"  {'─' * 58}")
print(f"  {'Writes per document':22} {'2':>12} {'1':>13} {'':>8}")
print(f"  {'Insertion time (ms)':22} {multi_insert_ms:>12.1f} {single_insert_ms:>13.1f} {insert_factor:>7.1f}x")
print(f"  {'Queries per retrieval':22} {'2 + join':>12} {'1':>13} {'':>8}")
print(f"  {'Avg retrieval (ms)':22} {multi_query_ms:>12.1f} {single_query_ms:>13.1f} {query_factor:>7.1f}x")

for col_name in ["multi_embed", "multi_doc", "single_arch"]:
    database[col_name].drop()

## 8. HNSW Parameter Tuning: Recall vs. Latency
Similar to above, this section makes use of documents already processed in previous cells to compare how HNSW index parameters affect the performance and accuracy of vector queries. 

HNSW has two build-time parameters (`m` and `efConstruction`) that require a full index rebuild to change, and one query-time parameter (`efSearch`) that can be tuned per query:

| Parameter | When Set | What It Controls |
|-----------|----------|-----------------|
| `m` | Index build | Connections per node. Higher = better recall, more memory |
| `efConstruction` | Index build | How many candidates are evaluated when inserting a node. Higher = better graph quality, slower build |
| `efSearch` | Per query | How deeply the graph is traversed at search time. Raise it to recover recall, lower it to cut latency |

Three different tests are performed using configurations for each of the index parameters:
- `Low` - minimal connections, tiny candidate pool (fast build, poor recall)
- `Recommended` - recommended production values
- `High` - dense connections, large candidate pool (slow build, high recall)

To measure recall we need to know the *correct* answer first. numpy computes exact cosine similarity across every document.
The K documents with the highest scores are the true nearest neighbours.
The HNSW results will be compared against this list to see how many it got right.

The results show `Recall@K` - how many of the K true nearest neighbours did HNSW return?
- **100%** = HNSW found every document the exact search found.
- **40%**  = HNSW only returned 2 of the 5 - it missed 3 documents that were actually closer to the query than some of the ones it did return.

In [ ]:
sample_size = 500
print(f"Loading {sample_size} documents from collection...")
raw_docs = list(collection.find(
    {"vectorContent": {"$exists": True}},
    {"textContent": 1, "vectorContent": 1}
).limit(sample_size))
print(f"  Loaded {len(raw_docs)} documents\n")

# Establish baseline via exact cosine similarity 
test_query = "What HNSW parameters should I configure for DocumentDB vector search?"
q_vec = np.array(embeddings.embed_query(test_query), dtype=np.float32)
corpus_matrix = np.array([d["vectorContent"] for d in raw_docs], dtype=np.float32)

cos_sims = (
    corpus_matrix / np.linalg.norm(corpus_matrix, axis=1, keepdims=True)
) @ (q_vec / np.linalg.norm(q_vec))

K = 5
true_top_ids = {raw_docs[i]["_id"] for i in np.argsort(cos_sims)[::-1][:K]}
print(f"Baseline: top {K} exact nearest neighbours identified\n")

# Configurations
index_configs = [
    {"label": "Low          (m=2,  efC=4)", "m": 2,  "efConstruction": 4  },
    {"label": "Recommended  (m=16, efC=64)", "m": 16, "efConstruction": 64 },
    {"label": "High         (m=64, efC=256)", "m": 64, "efConstruction": 256},
]

## efSearch=1   - traverse almost nothing: fastest possible query, lowest recall
## efSearch=64  - recommended default
## efSearch=512 - deep traversal: slowest query, highest recall
ef_search_values = [1, 64, 512]

# Build each index and benchmark
rows = []
for cfg in index_configs:
    demo_col = database[f"hnsw_tune_{cfg['m']}_{cfg['efConstruction']}"]
    demo_col.drop()
    demo_col.insert_many([
        {"_id": d["_id"], "textContent": d["textContent"], "vectorContent": d["vectorContent"]}
        for d in raw_docs
    ])

    t0 = time.perf_counter()
    demo_col.create_index(
        [("vectorContent", "vector")],
        vectorOptions={
            "type": "hnsw",
            "similarity": "cosine",
            "dimensions": vector_dimensions,
            "m": cfg["m"],
            "efConstruction": cfg["efConstruction"]
        },
        name="hnsw"
    )
    build_ms = (time.perf_counter() - t0) * 1000

    for ef in ef_search_values:
        t0 = time.perf_counter()
        hits = list(demo_col.aggregate([
            {"$search": {"vectorSearch": {
                "vector": q_vec.tolist(),
                "path": "vectorContent",
                "similarity": "cosine",
                "k": K,
                "efSearch": ef
            }}},
            {"$project": {"_id": 1}}
        ]))
        query_ms = (time.perf_counter() - t0) * 1000

        recall = len({h["_id"] for h in hits} & true_top_ids) / len(true_top_ids)
        rows.append((cfg["label"], ef, build_ms, query_ms, recall))

    demo_col.drop()

# Results table
print(f"  {'Index config':<32} {'efSearch':>8} {'Build(ms)':>10} {'Query(ms)':>10} {'Recall@5':>9}")
print("  " + "─" * 74)
prev_label = None
for label, ef, build_ms, query_ms, recall in rows:
    if prev_label and label != prev_label:
        print()
    print(f"  {label:<32} {ef:>8} {build_ms:>10.0f} {query_ms:>10.1f} {recall:>8.0%}")
    prev_label = label

## 9. Configure LLM and Vector Store
Initialize Claude Haiku as the LLM and set up the DocumentDB vector store for similarity search.

In [ ]:
# LLM for generating responses from retrieved context
llm = ChatBedrockConverse(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0", client=bedrock_client)

# Vector store wraps DocumentDB collection for similarity search
# Uses the HNSW index created earlier for fast nearest neighbor lookup
vector_store = DocumentDBVectorSearch(
    collection=collection,
    embedding=embeddings,
    index_name=index_name
)

print("llm and vector store configured.")

## 10. Chunk Overlap - Why It Matters
This section demonstrates how chunk overlap preserves context at boundaries. 
It inserts the same document chunked with and without overlap into a temp collection, then queries both.

When documents are split into chunks, information can be lost at boundaries. If the first chunk defines key terms and the second chunk refers to them only by pronoun ("the first parameter", "the second parameter"), the second chunk becomes ambiguous in isolation. Overlap carries the definitions forward so the LLM can correctly resolve references. It's not a nice-to-have — it prevents information loss at chunk boundaries.

The results show that without overlap, Chunk B is ambiguous - the LLM can't answer "what values should I use for m and efConstruction?"
However With overlap, the definitions carry forward into Chunk B, and the LLM correctly maps values to parameter names.

In [ ]:
custom_text = (
    "Amazon DocumentDB supports vector search using HNSW indexes. When building "
    "an HNSW index, two parameters control its quality: m sets the number of "
    "bi-directional links per node, and efConstruction sets the candidate pool "
    "size during construction. Tuning these correctly is critical. "
    "The first parameter should be set between 2 and 100, with 16 being optimal "
    "for most workloads. The second parameter should be set to four times the "
    "first, so the recommended configuration is 16 and 64 respectively. Setting "
    "the second too low causes poor graph connectivity and significantly degrades "
    "recall accuracy during similarity searches."
)

# Split right after the named definitions, before the pronoun-only section
split_phrase = "The first parameter should"
split_point = custom_text.index(split_phrase)

# Overlap carries forward: "m sets... efConstruction sets... Tuning these correctly is critical."
overlap_phrase = "m sets the number of bi-directional links per node, and efConstruction sets"
overlap_start = custom_text.index(overlap_phrase)

# WITHOUT overlap: Chunk B only has "first parameter", "second parameter" - no names
no_overlap_a = custom_text[:split_point].strip()
no_overlap_b = custom_text[split_point:].strip()

# WITH overlap: Chunk B includes the definitions of m and efConstruction
with_overlap_a = custom_text[:split_point].strip()
with_overlap_b = custom_text[overlap_start:].strip()

print("-" * 60)
print("Original document")
print("-" * 60)
print(custom_text)

print("\n" + "-" * 60)
print("WITHOUT overlap")
print("-" * 60)
print(f"\n  CHUNK A ({len(no_overlap_a)} chars):")
print(f"  {no_overlap_a}")
print(f"\n  CHUNK B ({len(no_overlap_b)} chars):")
print(f"  {no_overlap_b}")

print("\n" + "-" * 60)
print(f"WITH overlap ({split_point - overlap_start} char overlap)")
print("-" * 60)
print(f"\n  CHUNK A ({len(with_overlap_a)} chars):")
print(f"  {with_overlap_a}")
print(f"\n  CHUNK B ({len(with_overlap_b)} chars):")
print(f"  {with_overlap_b}")

# Embed all 4 chunks
all_texts = [no_overlap_a, no_overlap_b, with_overlap_a, with_overlap_b]
all_vectors = embeddings.embed_documents(all_texts)

temp_col = database["chunk_overlap_demo"]
temp_col.drop()
temp_col.insert_many([
    {"textContent": no_overlap_a, "vectorContent": all_vectors[0], "group": "no_overlap", "chunk": "A"},
    {"textContent": no_overlap_b, "vectorContent": all_vectors[1], "group": "no_overlap", "chunk": "B"},
    {"textContent": with_overlap_a, "vectorContent": all_vectors[2], "group": "overlap", "chunk": "A"},
    {"textContent": with_overlap_b, "vectorContent": all_vectors[3], "group": "overlap", "chunk": "B"},
])

temp_col.create_index(
    [("vectorContent", "vector")],
    vectorOptions={"type": "hnsw", "similarity": "cosine",
                   "dimensions": vector_dimensions, "m": 16, "efConstruction": 64},
    name="hnsw"
)

# Query asking for specific parameter names and values
search_query = "What values should I use for m and efConstruction in DocumentDB?"
query_vector = embeddings.embed_query(search_query)

print("\n" + "-" * 60)
print(f'QUERY: "{search_query}"')
print("-" * 60)

# Single search, pick best result per group
all_results = list(temp_col.aggregate([
    {"$search": {
        "vectorSearch": {"vector": query_vector, "path": "vectorContent",
                        "similarity": "cosine", "k": 4}
    }},
    {"$project": {"textContent": 1, "group": 1, "chunk": 1}}
]))

best = {}
for doc in all_results:
    g = doc["group"]
    if g not in best:
        best[g] = doc

for gk, label in [("no_overlap", "WITHOUT overlap"), ("overlap", "WITH overlap")]:
    if gk in best:
        doc = best[gk]
        print(f"\n  {label} - Best match comes from CHUNK {doc['chunk']}:")
        print(f"  {doc['textContent']}")

# Cleanup
temp_col.drop()

## 11. Prompt Template
Prompt that instructs LLM to answer DocumentDB questions using retrieved context.

In [ ]:
prompt_template = """You are a Q&A assistant specializing in Amazon DocumentDB.

Rules:
- Answer in natural, conversational prose. Do not use bullet points or numbered lists unless the user explicitly asks for a list.
- Be direct and concise. No preamble, no filler.
- Never reference or mention source materials. Present information as your own knowledge.
- Do not use phrases like 'According to...' or 'Based on the provided context...'
- Include code examples when relevant, formatted in code blocks.
- If the question is vague, ask a clarifying question.
- If the context does not contain enough information to answer, say so directly.

Context:
{context}

Chat history:
{chat_history}

Question: {question}"""

prompt = PromptTemplate(
    template=prompt_template, 
    input_variables=["context", "question", "chat_history"]
)

print("Prompt setup complete")

## 12. RAG vs No-RAG Comparison
Side-by-side comparison of the same question answered two ways:

- *Without RAG* — The LLM relies solely on its training data. It produces generic relational-to-NoSQL advice with no DocumentDB-specific guidance.
- *With RAG* — The question is first converted to a vector and used to retrieve the most relevant chunks from DocumentDB via similarity search. Those chunks are injected into the prompt as context, so the LLM grounds its answer in the actual ingested documentation.

This is the core value of RAG: the LLM's knowledge is supplemented with your own curated content at query time, without retraining the model.

In [ ]:
# Migration query - targets content from the ingested data modeling guide and developer guide.
demo_query = (
    "How should I approach migrating an existing relational schema to Amazon DocumentDB? "
    "What are the specific document modeling patterns I should use, and which relational "
    "constructs have no direct equivalent?"
)

# Without RAG
no_rag_response = llm.invoke(demo_query)

# With RAG
docs = vector_store.similarity_search(demo_query, k=5)
context = "\n\n".join(doc.page_content for doc in docs)
rag_response = (prompt | llm).invoke({
    "context": context,
    "question": demo_query,
    "chat_history": ""
})

# Build source list for display
sources = []
for i, doc in enumerate(docs, 1):
    meta = doc.metadata if isinstance(doc.metadata, dict) else {}
    source = meta.get("source", meta.get("metadata", {}).get("source", "N/A"))
    page = meta.get("page", meta.get("metadata", {}).get("page", ""))
    page_info = f" (page {page})" if page != "" else ""
    sources.append(f"{i}. `{source}`{page_info}")
sources_md = "\n".join(sources)

display(Markdown(f"""
## Query
> {demo_query}

---

## Without RAG - LLM training data only

{no_rag_response.content}

---

## With RAG - currated content retrieved from DocumentDB

**Retrieved sources:**

{sources_md}

**Response:**

{rag_response.content}
"""))

## 13. Chat Configuration
History length and document retrieval limits for the chat session.

In [ ]:
max_history_length = 6
max_docs = 5

print("Chat history setup complete")

## 14. Launch Chatbot
Launches the Gradio chat interface and provides the access URL. Open the link in a new window to start interacting with the chat agent.

In [ ]:
# Similarity search - the 'R' in RAG (Retrieval)
# Converts query to vector, finds k nearest documents using HNSW index
@lru_cache(maxsize=100)
def cached_similarity_search(query):
    try:
        return vector_store.similarity_search(query, k=max_docs)
    except Exception as e:
        print(f"Error in similarity search: {e}")
        return []

@CircuitBreaker(failure_threshold=5, reset_timeout=60)
@RateLimiter(max_requests=5, time_window=60)
@retry(
    retry=retry_if_exception_type(Exception),
    wait=wait_exponential(multiplier=1, min=4, max=10),
    stop=stop_after_attempt(3)
)
def query_data(query, history):
    try:
        chat_history = history or []
        if len(chat_history) > max_history_length * 2:
            chat_history = chat_history[-max_history_length * 2:]

        history_str = ""
        for msg in chat_history:
            if isinstance(msg, dict):
                role = "Human" if msg["role"] == "user" else "Assistant"
                history_str += f"{role}: {msg['content']}\n"
            elif isinstance(msg, (list, tuple)) and len(msg) == 2:
                if msg[0]:
                    history_str += f"Human: {msg[0]}\n"
                if msg[1]:
                    history_str += f"Assistant: {msg[1]}\n"

        context = "\n".join(
            doc.page_content for doc in cached_similarity_search(query)
        )

        output = (prompt | llm).invoke({
            "context": context,
            "question": query,
            "chat_history": history_str,
        })

        return output if isinstance(output, str) else output.content

    except Exception as e:
        print(f"Error occurred: {e}")
        return f"Sorry, an error occurred: {str(e)}"

icon_path = os.path.join(os.getcwd(), "assets/docdb_id_image.png")
if not os.path.exists(icon_path):
    icon_path = None

custom_css = """
* {
    font-family: 'Consolas', 'Monaco', 'Courier New', monospace !important;
}

/* Chat messages */
.message-wrap {
    font-size: 22px !important;
    line-height: 1.5 !important;
    padding: 15px !important;
}

/* User messages */
.message.user {
    font-size: 22px !important;
    font-weight: 500 !important;
}

/* Bot responses */
.message.bot {
    font-size: 22px !important;
    font-weight: 400 !important;
}

/* Input text area */
textarea {
    font-size: 22px !important;
    line-height: 1.5 !important;
}

.gr-textbox textarea {
    font-size: 22px !important;
    line-height: 1.5 !important;
}

input[type="text"] {
    font-size: 22px !important;
    line-height: 1.5 !important;
}

/* Chat interface input */
.chat-interface textarea {
    font-size: 22px !important;
    line-height: 1.5 !important;
}

/* Headers if any */
.title-text {
    font-size: 22px !important;
    font-weight: 600 !important;
}

/* Header container styling */
.header-container {
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    gap: 20px !important;
    padding: 20px !important;
}

/* Logo image styling */
.logo-image {
    width: 150px !important;
    height: 150px !important;
    object-fit: contain !important;
}

/* Title styling */
.header-title {
    font-size: 22px !important;
    margin: 0 !important;
}

.gradio-container {
    max-width: 100% !important;
    height: 100vh !important;
    display: flex !important;
    flex-direction: column !important;
}

.header-container {
    flex: 0 0 auto !important;
}

.chatbox {
    flex: 1 1 auto !important;
    min-height: 0 !important;
    height: calc(80vh - 200px) !important;
    overflow-y: auto !important;
}

.input-row {
    flex: 0 0 auto !important;
    position: sticky !important;
    bottom: 0 !important;
    background: white !important;
    padding: 10px !important;
    margin-top: auto !important;
}

/* Chat messages - broader selectors */
.chatbot .message {
    font-size: 22px !important;
    line-height: 1.5 !important;
}

.chatbot div {
    font-size: 22px !important;
}

/* All text in chatbot area */
.chatbot * {
    font-size: 22px !important;
}

/* Title - broader selectors */
h1, .gr-markdown h1 {
    font-size: 22px !important;
    font-weight: 600 !important;
}

/* Markdown content in header */
.gr-markdown {
    font-size: 22px !important;
}

/* All text content */
.gr-chatbot .message-wrap * {
    font-size: 22px !important;
}

/* ChatInterface description */
.gr-chatinterface .description,
.chatinterface-description,
.gr-markdown p {
    font-size: 32px !important;
    font-weight: 500 !important;
}

/* All paragraph text */
p {
    font-size: 32px !important;
}
"""

# Close any previously running Gradio server before launching a new one.
if "demo" in globals() and demo:
    demo.close()

with gr.Blocks() as demo:
    with gr.Row(elem_classes="header-container"):
        gr.Image(icon_path, show_label=False, height=150, width=150, elem_classes="logo-image")
        gr.Markdown("# DocumentDB SSA - ChatBot", elem_classes="header-title")

    with gr.Column(scale=1):
        bot = gr.Chatbot(
            container=True,
            show_label=False,
            elem_classes="chatbox"
        )

    chat_interface = gr.ChatInterface(
        fn=query_data,
        chatbot=bot,
        title="",
        description="Ask me questions about Amazon DocumentDB!",
        examples=[
            "What is Amazon DocumentDB?",
            "How does vector search work in DocumentDB?",
            "What are the pricing options for DocumentDB?"
        ]
    )

demo.launch(
    share=True,
    debug=True,
    server_name="0.0.0.0",
    server_port=7861,
    width="100%",
    css=custom_css,
    theme=gr.themes.Soft()
)

## 15. Cleanup
Close the DocumentDB connection when done.

In [ ]:
if 'client' in globals() and client:
    client.close()
    print("DocumentDB connection closed")